In [7]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
df = pd.read_csv('mostCommonExercises.csv')

print("Dataset Overview:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nTarget variable distribution:")
print(df['muscle_size_increase_cm2'].describe())

Dataset Overview:
         exercise_name target_muscle_group      type  age gender  sets  reps  \
0  Barbell Bench Press               Chest  Compound   32      M     4    10   
1  Barbell Bench Press               Chest  Compound   28      F     3    12   
2  Barbell Bench Press               Chest  Compound   39      M     5     8   
3  Barbell Bench Press               Chest  Compound   35      F     4    10   
4  Barbell Bench Press               Chest  Compound   26      M     3    12   

   weight  frequency  protein  calories  sleep    experience  \
0      80          3      150      2800    7.5  Intermediate   
1      45          2      130      2600    7.2      Beginner   
2      90          4      170      3100    8.0      Advanced   
3      50          3      135      2700    7.4  Intermediate   
4      85          2      145      2750    7.6  Intermediate   

   muscle_size_increase_cm2  genetic_advantage  
0                      2.50                  3  
1                 

In [9]:
# Define baseline muscle growth equation parameters with current size and diminishing returns
def get_baseline_params(age, gender, experience, current_size_cm, workout_time_years):
    if gender == 'M':
        base_limit = 45
        age_factor = max(0.7, 1 - (age - 25) * 0.01)
    else:
        base_limit = 30
        age_factor = max(0.7, 1 - (age - 25) * 0.008)
    M_max = base_limit * age_factor
    remaining_potential = max(0, M_max - (current_size_cm * 0.2))
    k_base = {'Beginner': 0.20, 'Intermediate': 0.10, 'Advanced': 0.05}
    k = k_base.get(experience, 0.10) * (1 / (1 + workout_time_years * 0.1))
    return remaining_potential, k

def calculate_baseline_growth(row, time_months=3, current_size_cm=0, workout_time_years=0):
    M_max, k = get_baseline_params(row['age'], row['gender'], row['experience'], current_size_cm, workout_time_years)
    M0 = current_size_cm * 0.2
    baseline_growth = (M_max - M0) * (1 - np.exp(-k * time_months))
    baseline_cm2 = baseline_growth * 5
    return baseline_cm2

# Calculate baseline growth
df['baseline_growth'] = df.apply(calculate_baseline_growth, axis=1)

# Calculate adjustment factor
df['adjustment_factor'] = df['muscle_size_increase_cm2'] / df['baseline_growth'].replace(0, np.finfo(float).eps)

print("\nBaseline vs Actual Growth:")
print(f"Mean baseline growth: {df['baseline_growth'].mean():.2f} cm2")
print(f"Mean actual growth: {df['muscle_size_increase_cm2'].mean():.2f} cm2")
print(f"Mean adjustment factor: {df['adjustment_factor'].mean():.2f}")

# Feature engineering
le_gender = LabelEncoder()
le_exercise = LabelEncoder()
le_muscle_group = LabelEncoder()
le_category = LabelEncoder()
le_experience = LabelEncoder()

df['gender_encoded'] = le_gender.fit_transform(df['gender'])
df['exercise_name_encoded'] = le_exercise.fit_transform(df['exercise_name'])
df['muscle_group_encoded'] = le_muscle_group.fit_transform(df['target_muscle_group'])
df['category_encoded'] = le_category.fit_transform(df['type'])
df['experience_encoded'] = le_experience.fit_transform(df['experience'])

df['protein_per_kg'] = df['protein'] / (df['weight'] * 0.45)
df['volume'] = df['sets'] * df['reps']
df['intensity'] = df['weight'] / df['reps']
df['calories_per_kg'] = df['calories'] / (df['weight'] * 0.45)

# Select features for ML model
features = [
    'age', 'gender_encoded', 'exercise_name_encoded', 'sets', 'reps', 'weight',
    'frequency', 'protein', 'calories', 'sleep', 'experience_encoded',
    'muscle_group_encoded', 'category_encoded', 'protein_per_kg', 'volume',
    'intensity', 'calories_per_kg'
]

X = df[features]
y = df['adjustment_factor']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Ensure no inf or NaN in train/test sets
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(X_train.mean())
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(X_test.mean())

# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred = rf_model.predict(X_test)

# Evaluate model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nModel Performance:")
print(f"Mean Squared Error: {mse:.4f}")
print(f"R² Score: {r2:.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))


Baseline vs Actual Growth:
Mean baseline growth: 47.45 cm2
Mean actual growth: 1.88 cm2
Mean adjustment factor: 0.05


ValueError: Input X contains infinity or a value too large for dtype('float32').

In [ ]:
# Function to make predictions for multiple workouts
def predict_muscle_growth(workouts, current_size_cm, workout_time_years):
    results = []
    for workout in workouts:
        # Create input DataFrame for the workout
        input_data = pd.DataFrame({
            'age': [workout['age']],
            'gender': [workout['gender']],
            'exercise_name': [workout['exercise_name']],
            'sets': [workout['sets']],
            'reps': [workout['reps']],
            'weight': [workout['weight']],
            'frequency': [workout['frequency']],
            'protein': [workout['protein']],
            'calories': [workout['calories']],
            'sleep': [workout['sleep']],
            'experience': [workout['experience']],
            'target_muscle_group': [workout['target_muscle_group']],
            'type': [workout['exercise_category']]
        })

        # Calculate baseline growth
        baseline = calculate_baseline_growth(input_data.iloc[0], current_size_cm=current_size_cm, workout_time_years=workout_time_years)

        # Encode categorical variables
        input_data['gender_encoded'] = le_gender.transform(input_data['gender'])
        input_data['exercise_name_encoded'] = le_exercise.transform(input_data['exercise_name'])
        input_data['muscle_group_encoded'] = le_muscle_group.transform(input_data['target_muscle_group'])
        input_data['category_encoded'] = le_category.transform(input_data['type'])
        input_data['experience_encoded'] = le_experience.transform(input_data['experience'])

        # Create additional features
        input_data['protein_per_kg'] = input_data['protein'] / (input_data['weight'] * 0.45)
        input_data['volume'] = input_data['sets'] * input_data['reps']
        input_data['intensity'] = input_data['weight'] / input_data['reps']
        input_data['calories_per_kg'] = input_data['calories'] / (input_data['weight'] * 0.45)

        # Get features for prediction
        X_input = input_data[features]

        # Predict adjustment factor
        adjustment = rf_model.predict(X_input)[0]

        # Final prediction
        final_prediction = baseline * adjustment if baseline > 0 else 0

        results.append({
            'exercise_name': workout['exercise_name'],
            'baseline_growth': baseline,
            'adjustment_factor': adjustment,
            'final_prediction': final_prediction
        })
    return results

In [ ]:
# Example usage of the prediction function with multiple workouts
workouts = [
    {
        'age': 28,
        'gender': 'M',
        'exercise_name': 'Barbell Bench Press',
        'sets': 4,
        'reps': 10,
        'weight': 80,
        'frequency': 3,
        'protein': 150,
        'calories': 2800,
        'sleep': 7.5,
        'experience': 'Intermediate',
        'target_muscle_group': 'Chest',
        'exercise_category': 'Compound'
    },
    {
        'age': 28,
        'gender': 'M',
        'exercise_name': 'Incline Barbell Bench Press',
        'sets': 3,
        'reps': 12,
        'weight': 70,
        'frequency': 2,
        'protein': 145,
        'calories': 2750,
        'sleep': 7.6,
        'experience': 'Intermediate',
        'target_muscle_group': 'Chest',
        'exercise_category': 'Compound'
    }
]

result = predict_muscle_growth(workouts, current_size_cm=30, workout_time_years=2)

print("Example Prediction Results:")
for r in result:
    print(f"Exercise: {r['exercise_name']}")
    print(f"Baseline Growth: {r['baseline_growth']:.2f} cm2")
    print(f"Adjustment Factor: {r['adjustment_factor']:.2f}")
    print(f"Final Prediction: {r['final_prediction']:.2f} cm2\n")

In [ ]:
# Visualizations
plt.figure(figsize=(15, 10))

# Plot 1: Baseline vs Actual
plt.subplot(2, 3, 1)
plt.scatter(df['baseline_growth'], df['muscle_size_increase_cm2'], alpha=0.6)
plt.plot([df['baseline_growth'].min(), df['baseline_growth'].max()],
         [df['baseline_growth'].min(), df['baseline_growth'].max()], 'r--')
plt.xlabel('Baseline Growth (cm²)')
plt.ylabel('Actual Growth (cm²)')
plt.title('Baseline vs Actual Growth')

# Plot 2: Adjustment factors
plt.subplot(2, 3, 2)
plt.hist(df['adjustment_factor'], bins=20, alpha=0.7)
plt.xlabel('Adjustment Factor')
plt.ylabel('Frequency')
plt.title('Distribution of Adjustment Factors')

# Plot 3: Predicted vs Actual
plt.subplot(2, 3, 3)
y_pred_full = rf_model.predict(X)
plt.scatter(y, y_pred_full, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
plt.xlabel('Actual Adjustment Factor')
plt.ylabel('Predicted Adjustment Factor')
plt.title('Model Predictions vs Actual')

# Plot 4: Feature importance
plt.subplot(2, 3, 4)
top_features = feature_importance.head(8)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('Top Feature Importances')

# Plot 5: Final predictions
plt.subplot(2, 3, 5)
final_predictions = df['baseline_growth'] * y_pred_full
plt.scatter(df['muscle_size_increase_cm2'], final_predictions, alpha=0.6)
plt.plot([df['muscle_size_increase_cm2'].min(), df['muscle_size_increase_cm2'].max()],
         [df['muscle_size_increase_cm2'].min(), df['muscle_size_increase_cm2'].max()], 'r--')
plt.xlabel('Actual Growth (cm²)')
plt.ylabel('Final Predicted Growth (cm²)')
plt.title('Final Predictions vs Actual')

# Plot 6: Residuals
plt.subplot(2, 3, 6)
residuals = df['muscle_size_increase_cm2'] - final_predictions
plt.scatter(final_predictions, residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Growth (cm²)')
plt.ylabel('Residuals')
plt.title('Residuals Plot')

plt.tight_layout()
plt.show()

print("\nModel Training Complete!")
print("The model learns to adjust the baseline muscle growth equation based on individual factors and current size.")
print("Use the predict_muscle_growth() function with an array of workouts to make new predictions.")